# Perceptron Classification on MNIST

This notebook implements an end-to-end machine learning pipeline using a Perceptron classifier on the MNIST dataset.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.linear_model import Perceptron
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
possible_paths = [
    Path("datasets/mnist.csv"),
    Path("../datasets/mnist.csv"),
    Path("../../datasets/mnist.csv"),
    Path("mnist.csv"),
]

data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "mnist.csv was not found. Place it in this folder or inside a datasets folder."
    )

mnist = pd.read_csv(data_path)

print("Dataset path:", data_path)
print("Dataset shape:", mnist.shape)
mnist.head()


In [ ]:
target_candidates = ["label", "target", "class", "digit"]
target_column = next(
    (column for column in target_candidates if column in mnist.columns),
    mnist.columns[0],
)

X = mnist.drop(columns=target_column)
y = mnist[target_column]

print("Target column:", target_column)
print("Features:", X.shape)
print("Target:", y.shape)
print(y.value_counts().sort_index())


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for index, axis in enumerate(axes.ravel()):
    axis.imshow(X.iloc[index].to_numpy().reshape(28, 28), cmap="gray")
    axis.set_title(f"Label: {y.iloc[index]}")
    axis.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


In [ ]:
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "perceptron",
            Perceptron(
                random_state=42,
                early_stopping=True,
                validation_fraction=0.10,
            ),
        ),
    ]
)

pipeline


In [ ]:
parameter_grid = {
    "perceptron__penalty": [None, "l2", "l1", "elasticnet"],
    "perceptron__alpha": [0.0001, 0.001],
    "perceptron__eta0": [0.1, 1.0],
    "perceptron__max_iter": [1000, 2000],
}

search = GridSearchCV(
    pipeline,
    parameter_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

print("Best CV accuracy:", round(search.best_score_, 4))
print("Best parameters:")
search.best_params_


In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

results = pd.Series(
    {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (weighted)": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
)

results.round(4)


In [ ]:
report = pd.DataFrame(
    classification_report(y_test, y_pred, output_dict=True, zero_division=0)
).transpose()

report.round(4)


In [ ]:
fig, axis = plt.subplots(figsize=(9, 9))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    cmap="Blues",
    colorbar=False,
    ax=axis,
)

axis.set_title("Perceptron Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
trained_perceptron = best_model.named_steps["perceptron"]

{
    "Classes": trained_perceptron.classes_,
    "Coefficient shape": trained_perceptron.coef_.shape,
    "Intercept shape": trained_perceptron.intercept_.shape,
    "Iterations": trained_perceptron.n_iter_,
    "Training samples observed": trained_perceptron.t_,
}


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))

for index, (digit, axis) in enumerate(zip(trained_perceptron.classes_, axes.ravel())):
    axis.imshow(trained_perceptron.coef_[index].reshape(28, 28), cmap="coolwarm")
    axis.set_title(f"Digit {digit}")
    axis.axis("off")

plt.suptitle("Perceptron Coefficients by Digit")
plt.tight_layout()
plt.show()


## Conclusion

The Perceptron was trained in a Scikit-learn pipeline, optimised through cross-validation, and evaluated on an unseen test set. The reported metrics, classification report, confusion matrix, and model attributes summarise its performance.
